In [ ]:
import scanpy as sc
import pandas as pd
import scanpy as sc

In [ ]:
adata = sc.read_h5ad("/workspace/data/de122_lce75/adata_de122_lce75_merged.h5ad")

In [ ]:
from essential.data import load_regulondb_full
from run_prediction import compute_amask

ref_db = load_regulondb_full()
ref_db = ref_db.loc[lambda x: x["ri_type"].str.startswith("TF")]

_, gene_is_regulator, gene_is_target = compute_amask(adata, ref_db)
valid_features = gene_is_regulator | gene_is_target
adata = adata[:, valid_features].copy()
Amask, _, _ = compute_amask(adata, ref_db)

In [ ]:
metrics = pd.read_csv("/workspace/experiments/05152026_cellbox_comeback/metrics.csv")
metrics

In [ ]:
adata.var_names[adata.var_names.str.lower().str.contains("lac")]

In [ ]:
Amask_df = pd.DataFrame(Amask, index=adata.var_names, columns=adata.var_names)
Amask_df # (targets, regulators)

In [ ]:
number_of_regulated_genes = Amask_df.sum(0).to_frame("number_of_regulated_genes")
number_of_regulated_genes

In [ ]:
Amask_df.loc[["lacZ"]].loc[:, ["lacI"]]

In [ ]:
mse_ratios = (
    metrics.pivot_table(index="target", values="mse_overall", columns=["model_name"])
    .reset_index()
    .assign(mse_ratio=lambda df: df["cellbox"] / df["mean_baseline"])
    .sort_values("mse_ratio", ascending=True)
    .merge(number_of_regulated_genes, left_on="target", right_index=True)
)
mse_ratios.head(25)
# print("best predicted targets:")
# print(", ".join(mse_ratios.head(25)["target"]))

In [ ]:
mse_ratios.tail(25)